In [0]:
data = [
    ("HR", 50000),
    ("IT", 75000),
    ("Finance", 60000),
    ("Marketing", 55000),
    ("HR", 52000),
    ("IT", 80000),
    ("Finance", 65000),
    ("Marketing", 58000),
    ("HR", 48000),
    ("IT", 72000),
]

df = spark.createDataFrame(data, ["dept", "salary"])
df.display()
df.createOrReplaceTempView("emp")

dept,salary
HR,50000
IT,75000
Finance,60000
Marketing,55000
HR,52000
IT,80000
Finance,65000
Marketing,58000
HR,48000
IT,72000


In [0]:
spark.sql("select * from (select dept,salary,dense_rank() over (partition by dept order by salary desc) rk from emp) where rk =2").display()

dept,salary,rk
Finance,60000,2
HR,50000,2
IT,75000,2
Marketing,55000,2


In [0]:
from pyspark.sql.functions import *
df_denserank = df.withColumn("rk",expr("dense_rank() over (partition by dept order by salary desc) rk"))
df_denserank.filter("rk = 2").drop("rk").display()

dept,salary
Finance,60000
HR,50000
IT,75000
Marketing,55000


In [0]:
from pyspark.sql.window import Window

# df_win = Window.partitionBy("dept").orderBy(col("salary").desc()) # column object if uses function like .desc() , .asc() , .alias() and Mathematical or string manipulation functions (e.g., col("salary") * 1.1)
df_win = Window.partitionBy("dept").orderBy(desc("salary"))
df_denserank = df.withColumn("rk",dense_rank().over (df_win))
df_denserank.filter("rk = 2").drop("rk").display()

dept,salary
Finance,60000
HR,50000
IT,75000
Marketing,55000


In [0]:
data = [
    (1, "Sai",   "Python",     30000, "ACTIVE"),
    (2, "Raj",   "Python",     32000, "ACTIVE"),
    (3, "John",  "Java",       28000, "ACTIVE"),
    (4, "Rita",  "Python",     31000, "ACTIVE"),
    (5, "Sam",   "Java",       29000, "ACTIVE"),
    (6, "Kiran", "Databricks", 33000, "ACTIVE"),
    (7, "Anu",   "Python",     27000, "ACTIVE"),
    (8, "Ram",   "Spark",      34000, "ACTIVE"),
    (9, "Priya", "Spark",      30000, "ACTIVE"),
    (10,"Vijay", "Databricks", 35000, "ACTIVE")
]

df_students = spark.createDataFrame(
    data,
    ["student_id", "name", "course", "fee", "status"]
)


data1 = [
    (5,  50000),
    (6,  50000),
    (8,  50000),
    (10, 50000)
]

df_updated_fees = spark.createDataFrame(
    data1,
    ["student_id", "fee"]
)

In [0]:
df_students.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("students")
df_updated_fees.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("updated_fees")

#df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("students")

In [0]:
%sql
select * from students order by 1;

student_id,name,course,fee,status
1,Sai,Python,30000,ACTIVE
2,Raj,Python,32000,ACTIVE
3,John,Java,28000,ACTIVE
4,Rita,Python,31000,ACTIVE
5,Sam,Java,29000,ACTIVE
6,Kiran,Databricks,33000,ACTIVE
7,Anu,Python,27000,ACTIVE
8,Ram,Spark,34000,ACTIVE
9,Priya,Spark,30000,ACTIVE
10,Vijay,Databricks,35000,ACTIVE


In [0]:
%sql
update students set fee = fee + 5000 where student_id=1;

num_affected_rows
1


In [0]:
from delta.tables import DeltaTable
deltaTable = DeltaTable.forName(spark, "students")
deltaTable.update(condition="student_id =1", set={"fee": "fee + 5000"})

DataFrame[num_affected_rows: bigint]

In [0]:
df = spark.read.format("delta").table("students")
df.display()

student_id,name,course,fee,status
3,John,Java,28000,ACTIVE
5,Sam,Java,29000,ACTIVE
6,Kiran,Databricks,33000,ACTIVE
7,Anu,Python,27000,ACTIVE
9,Priya,Spark,30000,ACTIVE
10,Vijay,Databricks,35000,ACTIVE
11,null,Python,99999,ACTIVE
12,null,Python,99999,ACTIVE
13,null,Java,99999,ACTIVE
2,Raj,Python,55555,ACTIVE


In [0]:
%sql
select * from updated_fees;

student_id,fee
5,50000
6,50000
8,50000
10,50000


In [0]:
%sql
merge into students as target
using updated_fees as source
on t.student_id = s.student_id
when matched then update set fee = s.fee
when not matched then insert (student_id, fee) values (s.student_id, s.fee)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7670162665312753>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'merge into students as target\nusing updated_fees as source\non t.student_id = s.student_id\nwhen matched then update set fee = s.fee\nwhen not matched then insert (student_id, fee) values (s.student_id, s.fee)\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /da

In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "students")

deltaTable.alias("target").merge(
    df_updated_fees.alias("source"), "t.student_id = s.student_id"
).whenMatchedUpdate(set={"fee": "s.fee"}).execute()


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(spark.table("students").orderBy("student_id"))

student_id,name,course,fee,status
1,Sai,Python,30000,ACTIVE
2,Raj,Python,32000,ACTIVE
3,John,Java,28000,ACTIVE
4,Rita,Python,31000,ACTIVE
5,Sam,Java,50000,ACTIVE
6,Kiran,Databricks,50000,ACTIVE
7,Anu,Python,27000,ACTIVE
8,Ram,Spark,50000,ACTIVE
9,Priya,Spark,30000,ACTIVE
10,Vijay,Databricks,50000,ACTIVE


In [0]:
data = [
    (1, "Sai",   "Python",     30000, "ACTIVE"),
    (2, "Raj",   "Python",     32000, "ACTIVE"),
    (3, "John",  "Java",       28000, "ACTIVE"),
    (4, "Rita",  "Python",     31000, "ACTIVE"),
    (5, "Sam",   "Java",       29000, "ACTIVE"),
    (6, "Kiran", "Databricks", 33000, "ACTIVE"),
    (7, "Anu",   "Python",     27000, "ACTIVE"),
    (8, "Ram",   "Spark",      34000, "ACTIVE"),
    (9, "Priya", "Spark",      30000, "ACTIVE"),
    (10,"Vijay", "Databricks", 35000, "ACTIVE")
]

df_students = spark.createDataFrame(
    data,
    ["student_id", "name", "course", "fee", "status"]
)


data1 = [
    (1, "PySpark",     55555, "ACTIVE"),
    (3, "PySpark",     55555, "ACTIVE"),
    (5, "PySpark",     55555, "ACTIVE"),
    (7, "PySpark",     55555, "ACTIVE"),
    (9, "PySpark",     55555, "ACTIVE"),
    
]

df_updated_mul_columns = spark.createDataFrame(
    data1,
    ["student_id", "course", "fee", "status"]
)

In [0]:
df_students.write.format("delta").mode("overwrite").saveAsTable("students")
df_updated_mul_columns.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("updated_mul_columns")

In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "students")

deltaTable.alias("target").merge(
    df_updated_mul_columns.alias("source"), "target.student_id = source.student_id"
).whenMatchedUpdate(
    set={"course": "source.course", "fee": "source.fee", "status": "source.status"}
).execute()

display(spark.table("students").orderBy("student_id"))

student_id,name,course,fee,status
1,Sai,PySpark,55555,ACTIVE
2,Raj,Python,32000,ACTIVE
3,John,PySpark,55555,ACTIVE
4,Rita,Python,31000,ACTIVE
5,Sam,PySpark,55555,ACTIVE
6,Kiran,Databricks,33000,ACTIVE
7,Anu,PySpark,55555,ACTIVE
8,Ram,Spark,34000,ACTIVE
9,Priya,PySpark,55555,ACTIVE
10,Vijay,Databricks,35000,ACTIVE


In [0]:
%sql

merge into students as target
using updated_mul_columns as source
on target.student_id = source.student_id
when matched then
  update set
    course = source.course,
    fee    = source.fee,
    status = source.status;

select * from students order by student_id;

student_id,name,course,fee,status
1,Sai,PySpark,55555,ACTIVE
2,Raj,Python,32000,ACTIVE
3,John,PySpark,55555,ACTIVE
4,Rita,Python,31000,ACTIVE
5,Sam,PySpark,55555,ACTIVE
6,Kiran,Databricks,33000,ACTIVE
7,Anu,PySpark,55555,ACTIVE
8,Ram,Spark,34000,ACTIVE
9,Priya,PySpark,55555,ACTIVE
10,Vijay,Databricks,35000,ACTIVE


In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "students")

deltaTable.alias("target").merge(
    df_updated_mul_columns.alias("source"), "target.student_id = source.student_id"
).whenMatchedDelete().execute()

display(spark.table("students").orderBy("student_id"))

student_id,course,fee,status
2,Python,32000,ACTIVE
4,Python,31000,ACTIVE
6,Databricks,33000,ACTIVE
8,Spark,34000,ACTIVE
10,Databricks,35000,ACTIVE


In [0]:
%sql
alter table students set tblproperties ('delta.columnMapping.mode' = 'name');
alter table students drop column name;

In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "students")

deltaTable.alias("target").merge(
    df_updated_mul_columns.alias("source"), "target.student_id = source.student_id"
).whenMatchedUpdateAll().execute()

display(spark.table("students").orderBy("student_id"))

student_id,course,fee,status
2,Python,32000,ACTIVE
4,Python,31000,ACTIVE
6,Databricks,33000,ACTIVE
8,Spark,34000,ACTIVE
10,Databricks,35000,ACTIVE
